# Gaussian Naive Bayes

In [1]:
import os
os.chdir("E:\Data Science\ML\Project")

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score)

## load the dataset

In [3]:
df = pd.read_csv("data\ChurnGuard_processed.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9259 entries, 0 to 9258
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                9259 non-null   int64  
 1   gender             9259 non-null   object 
 2   tenure_months      9259 non-null   int64  
 3   contract_type      9259 non-null   object 
 4   monthly_charges    9259 non-null   float64
 5   total_charges      9259 non-null   float64
 6   internet_service   9259 non-null   object 
 7   payment_method     9259 non-null   object 
 8   support_calls      9259 non-null   int64  
 9   late_payments      9259 non-null   int64  
 10  online_security    9259 non-null   object 
 11  tech_support       9259 non-null   object 
 12  streaming_service  9259 non-null   object 
 13  senior_citizen     9259 non-null   int64  
 14  family_members     9259 non-null   int64  
 15  churn              9259 non-null   int64  
dtypes: float64(2), int64(7),

In [5]:
X = df.drop(columns="churn")
y = df.churn

## Data Preprocessing & Gaussian Naive Bayes Pipeline

In [6]:
num_var = X.select_dtypes(exclude="object").columns
cat_var = X.select_dtypes(include="object").columns

cat_tran = Pipeline([("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))])
num_tran = Pipeline([("num", StandardScaler())])
preprocessing = ColumnTransformer([("num", num_tran, num_var), ("cat", cat_tran, cat_var)])
model = Pipeline([("preprocessing", preprocessing), ("classifier", GaussianNB())])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=47)

## Define GridSearch parameters

In [9]:
param_grid = {
    "classifier__var_smoothing": np.logspace(-12, -6, 20)
}

## GridSearchCV

In [10]:
grid_nb = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1, scoring="accuracy")
grid_nb.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,Pipeline(step...aussianNB())])
,param_grid,{'classifier__var_smoothing': array([1.0000...00000000e-06])}
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


## Best parameters

In [11]:
print("Best Parameters:")
print(grid_nb.best_params_)

print("\nBest CV Accuracy:")
print(grid_nb.best_score_)

Best Parameters:
{'classifier__var_smoothing': np.float64(1e-12)}

Best CV Accuracy:
0.7825030685580101


## Model Performance Evaluation